In [0]:
from pyspark.sql.functions import(col, concat, lit, to_timestamp, to_date,try_to_timestamp, concat_ws,trim, upper, lower, when,udf, col )
from pyspark.sql.types import LongType

In [0]:
%run "../includes/common_functions"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.silver;

In [0]:
results_df = spark.read.table("f1.bronze.results")

In [0]:
results_renamed_df = results_df\
    .withColumnRenamed("resultId", "result_id")\
    .withColumnRenamed("raceId", "race_id")\
    .withColumnRenamed("driverId","driver_id")\
    .withColumnRenamed("constructorId", "constructor_id")\
    .withColumnRenamed("positionText", "position_text")\
    .withColumnRenamed("fastestLap","fastest_lap")\
    .withColumnRenamed("milliseconds", "race_time_ms")\
    .withColumnRenamed("fastestLapTime", "fastest_lap_time")\
    .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed")\
    .withColumnRenamed("statusId", "status_id")\
    .withColumnRenamed("positionOrder", "position_order")

In [0]:
results_renamed_df = results_renamed_df \
    .withColumn("position_text", trim(col("position_text"))) \
    .withColumn("fastest_lap_time", trim(col("fastest_lap_time"))) \
    .withColumn("time", trim(col("time")))

In [0]:
results_clean_df = results_renamed_df \
    .filter(col("result_id").isNotNull()) \
    .dropDuplicates(["result_id"])

In [0]:
results_clean_df = results_clean_df.withColumn(
    "time",
    when(col("time") == "\\N", None)
    .otherwise(col("time"))
)

In [0]:
results_clean_df = results_clean_df.withColumn(
    "fastest_lap_time",
    when(col("fastest_lap_time") == "\\N", None)
    .otherwise(col("fastest_lap_time"))
)

In [0]:
results_clean_df = results_clean_df.withColumn(
    "fastest_lap_time_ms",
    convert_fastest_lap_udf(col("fastest_lap_time"))
)

In [0]:
display(results_clean_df)

In [0]:
results_final_df = add_ingestion_date(results_clean_df) \
    .withColumn("environment", lit("production")) \
    .withColumn("file_date", to_date(lit("2026-05-07")))

In [0]:
results_final_df.createOrReplaceTempView("results_updates")

In [0]:
if spark.catalog.tableExists("f1.silver.results"):

    spark.sql("""

    MERGE INTO f1.silver.results tgt
    USING results_updates src
    ON tgt.result_id = src.result_id

    WHEN MATCHED THEN
    UPDATE SET *

    WHEN NOT MATCHED THEN
    INSERT *

    """)

else:

    results_final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("f1.silver.results")

In [0]:
%sql

SELECT *
FROM f1.silver.results
LIMIT 10;